# Exploratory Data Analysis
This notebook will explore the dataset provided for the purposes of feature engineering

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
from collections import Counter
warnings.filterwarnings('ignore')

### Fetching training data from local store

In [ ]:
train_data_path = "data/train_df.csv"
train_df = pd.read_csv(train_data_path)
print(train_df.head())

### Identifying missing values

I have implemented two options, first is to replace the NA values with an empty string. The second is to drop the observation all together.

Importantly from the output below there are two rows that have NAs. Both rows have missing fake texts. Since the sample size is already small, injecting empty strings may influence performance of the algorithms more than removing the samples altogether. 

In [ ]:
clean_df = train_df.copy()
print(f"Number of samples before data cleaning: {len(clean_df)}")

# Method 1: Replacing NaNs with empty strings
for idx, row in clean_df.iterrows():
    if pd.isna(row["file_1"]):
        print(row)
        # clean_df.at[idx, "file_1"] = ""
    elif pd.isna(row["file_2"]):
        print(row)
        # clean_df.at[idx, "file_2"] = ""

# Method 2: Removing NA rows altogether 
clean_df.dropna(subset=["file_1", "file_2"], inplace=True)
print(f"Number of samples before data cleaning: {len(clean_df)}")

### Extract Basic Data Functions 

In [ ]:
def extract_basic_features(df):
    # Create real and fake text columns
    df['real_text'] = df.apply(
        lambda row: row['file_1'] if row['real_file_label'] == 1 else row['file_2'], 
        axis=1
    )
    df['fake_text'] = df.apply(
        lambda row: row['file_2'] if row['real_file_label'] == 1 else row['file_1'], 
        axis=1
    )

    features = []

    for idx, row in df.iterrows():
        real_text = row['real_text']
        fake_text = row['fake_text']
        
        # Basic length features
        real_chars = len(real_text)
        fake_chars = len(fake_text)
        real_words = len(real_text.split())
        fake_words = len(fake_text.split())
        
        # Basic punctuation features
        real_punct = len(re.findall(r'[.!?,:;]', real_text))
        fake_punct = len(re.findall(r'[.!?,:;]', fake_text))
        real_punct_types = len(set(re.findall(r'[.!?,:;()]', real_text)))
        fake_punct_types = len(set(re.findall(r'[.!?,:;()]', fake_text)))
        
        # Basic sentence features
        real_sents = len([s for s in re.split(r'[.!?]+', real_text) if s.strip()])
        fake_sents = len([s for s in re.split(r'[.!?]+', fake_text) if s.strip()])
        #real_avg_sent_len = np.mean([len(s.split()) for s in real_sents]) if real_sents else 0
        #fake_avg_sent_len = np.mean([len(s.split()) for s in fake_sents]) if fake_sents else 0
        
        # Basic variance features
        real_word_lengths = [len(w) for w in real_text.split()] if real_words > 0 else [0]
        fake_word_lengths = [len(w) for w in fake_text.split()] if fake_words > 0 else [0]
        real_word_var = np.var(real_word_lengths) if len(real_word_lengths) > 1 else 0
        fake_word_var = np.var(fake_word_lengths) if len(fake_word_lengths) > 1 else 0
        
        # Count proper nouns
        real_proper = len([w for i, w in enumerate(real_text.split()) 
                            if w and w[0].isupper() and i > 0])
        fake_proper = len([w for i, w in enumerate(fake_text.split()) 
                            if w and w[0].isupper() and i > 0])
        
        # Numbers
        real_numbers = len(re.findall(r'\d+', real_text))
        fake_numbers = len(re.findall(r'\d+', fake_text))
        real_precise_nums = len([n for n in re.findall(r'\d+\.?\d*', real_text) if '.' in n])
        fake_precise_nums = len([n for n in re.findall(r'\d+\.?\d*', fake_text) if '.' in n])
        real_large_nums = len([n for n in re.findall(r'\d+', real_text) if len(n) >= 3])
        fake_large_nums = len([n for n in re.findall(r'\d+', fake_text) if len(n) >= 3])

        # Science terms
        science_terms = ['telescope', 'survey', 'observation', 'stellar', 'galaxy', 'star', 
                'astronomical', 'magnitude', 'photometric', 'spectroscopic', 
                'wavelength', 'redshift', 'luminosity', 'parsec', 'light-year']

        real_science_count = sum(1 for term in science_terms if term.lower() in real_text.lower())
        fake_science_count = sum(1 for term in science_terms if term.lower() in fake_text.lower())

        # Abbreviations
        abbrevs = ['ESO', 'NASA', 'VLT', 'HST', 'ALMA', 'VISTA', 'VIRSA', 'VMC', 'VVV']
        real_abbrev_count = sum(1 for abbrev in abbrevs if abbrev in real_text)
        fake_abbrev_count = sum(1 for abbrev in abbrevs if abbrev in fake_text)

        # Repetition patterns
        real_word_freq = Counter(real_text.lower().split())
        fake_word_freq = Counter(fake_text.lower().split())
        real_repetition_score = sum(count for count in real_word_freq.values() if count > 1)
        fake_repetition_score = sum(count for count in fake_word_freq.values() if count > 1)
        real_unique_ratio = len(set(real_text.lower().split())) / (len(real_text.split()) + 1)
        fake_unique_ratio = len(set(fake_text.lower().split())) / (len(fake_text.split()) + 1)
        
        feature_row = {
            # Length differences
            'char_real': real_chars,
            'char_fake': fake_chars,
            'word_real': real_words,
            'word_fake': fake_words,
            'char_diff': real_chars - fake_chars,
            'word_diff': real_words - fake_words,
            'char_ratio': real_chars / (fake_chars + 1) if fake_chars > 0 else real_chars,
            'word_ratio': real_words / (fake_words + 1) if fake_words > 0 else real_words,
            
            # Punctuation differences  
            'punct_real': real_punct,
            'punct_fake': fake_punct, 
            'punct_diff': real_punct - fake_punct,
            'real_punct_types': real_punct_types,
            'fake_punct_types': fake_punct_types, 
            'punct_density_real': real_punct / (real_words + 1),
            'punct_density_fake': fake_punct / (fake_words + 1),
            'punct_density_diff': (real_punct / (real_words + 1)) - (fake_punct / (fake_words + 1)),
            
            # Sentence differences
            'real_sent': real_sents,
            'fake_sent': fake_sents,
            'sent_diff': real_sents - fake_sents,
            'sent_ratio': real_sents / (fake_sents + 1) if fake_sents > 0 else real_sents,
            # 'real_avg_sent_len': real_avg_sent_len,
            # 'fake_avg_sent_len': fake_avg_sent_len,

            
            # Variance differences
            'word_var_real': real_word_var,
            'word_var_fake': fake_word_var,
            'word_var_diff': real_word_var - fake_word_var,
            
            # Content differences
            'real_proper': real_proper,
            'fake_proper': fake_proper,
            'proper_diff': real_proper - fake_proper,
            'proper_density_real': real_proper / (real_words + 1),
            'proper_density_fake': fake_proper / (fake_words + 1),

            # Numbers
            'real_numbers': real_numbers,
            'fake_numbers': fake_numbers,
            'numbers_diff': real_numbers - fake_numbers,
            'real_precise_nums': real_precise_nums,
            'fake_precise_nums': fake_precise_nums,
            'real_large_nums': real_large_nums,
            'fake_large_nums': fake_large_nums,

            # Science terms
            'real_science_count': real_science_count,
            'fake_science_count': fake_science_count,

            # Abbreviations
            'real_abbrev_count': real_abbrev_count,
            'fake_abbrev_count': fake_abbrev_count,

            # Repetition patterns
            'real_repetition_score': real_repetition_score,
            'fake_repetition_score': fake_repetition_score,
            'real_unique_ratio': real_unique_ratio,
            'fake_unique_ratio': fake_unique_ratio
        }
        
        features.append(feature_row)

    return pd.DataFrame(features)

In [ ]:
features_df = extract_basic_features(clean_df)
labels = (clean_df['real_file_label'] - 1).values
print(features_df.describe())

In [ ]:
def analyze_feature_distributions(features_df):
    for feature in features_df.columns:
        values = features_df[feature]
        
        print(f"{feature}:")
        print(f"  Mean: {values.mean():.4f}")
        print(f"  Std:  {values.std():.4f}")
        print(f"  Min:  {values.min():.4f}")
        print(f"  Max:  {values.max():.4f}")
        
        # For difference features, show how often real > fake
        if 'diff' in feature:
            positive_count = (values > 0).sum()
            negative_count = (values < 0).sum()
            zero_count = (values == 0).sum()
            total = len(values)
            
            print(f"  Real > Fake: {positive_count}/{total} ({100*positive_count/total:.1f}%)")
            print(f"  Real < Fake: {negative_count}/{total} ({100*negative_count/total:.1f}%)")
            print(f"  Real = Fake: {zero_count}/{total} ({100*zero_count/total:.1f}%)")
        
        print()

analyze_feature_distributions(features_df)

In [ ]:
def compare_real_vs_fake_features(features_df):    
    # Compare pairs of real/fake features
    feature_pairs = [
        ('char_real', 'char_fake'),
        ('word_real', 'word_fake'),
        ('punct_real', 'punct_fake'),
        ('punct_density_real', 'punct_density_fake'),
        ('real_sent', 'fake_sent'),
        ('word_var_real', 'word_var_fake'),
        ('real_proper', 'fake_proper'),
        ('proper_density_real', 'proper_density_fake'),
        ('real_punct_types', 'fake_punct_types'),
        ('real_numbers', 'fake_numbers'),
        ('real_precise_nums', 'fake_precise_nums'),
        ('real_large_nums','fake_large_nums'),
        ('real_science_count', 'fake_science_count'),
        ('real_abbrev_count', 'fake_abbrev_count'),
        ('real_repetition_score', 'fake_repetition_score'),
        ('real_unique_ratio', 'fake_unique_ratio')
    ]
    
    for real_feature, fake_feature in feature_pairs:
        if real_feature in features_df.columns and fake_feature in features_df.columns:
            real_values = features_df[real_feature]
            fake_values = features_df[fake_feature]
            
            print(f"{real_feature} vs {fake_feature}:")
            print(f"  Real mean: {real_values.mean():.4f}")
            print(f"  Fake mean: {fake_values.mean():.4f}")
            print(f"  Difference: {real_values.mean() - fake_values.mean():.4f}")
            print()

compare_real_vs_fake_features(features_df)